# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates step-by-step data exploration and processing of the FAIR^2 dataset, which describes protein abundance, modifications, and sequences in human samples analyzed using mass spectrometry, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. Each entity—record set, field, and column—is referenced by its Croissant schema `@id` for precision and reproducibility.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

This dataset contains protein records as annotated by the experimenters, including accession, description, molecular weight, peptide counts, and calculated protein isoelectric point (pI) values.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR^2 dataset's metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded.")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
List all available record sets, their fields, and column `@id`s for transparent, schema-driven exploration.

In [ ]:
# Iterate over record sets and list their @id and fields
print("Available record sets and schema fields:")
record_sets = dataset.metadata.record_sets

for rs in record_sets:
    print(f"- Record set @id: {rs['@id']}")
    if 'field' in rs:
        for field in rs['field']:
            print(f"    - Field @id: {field['@id']} | name: {field.get('name','')} | dataType: {field.get('dataType','')}")
    print()

## 3. Data Extraction
Load records from a selected record set into a DataFrame. In this dataset, there is typically a single main record set containing the protein table.

**All Croissant objects are referenced via their `@id`.**

In [ ]:
# Identify the main protein record set (by inspection of record set @id above; adjust if record_set list is empty)

# If no record sets were found via the schema's metadata, fetch available record sets via dataset API
rs_ids = []
for rs in dataset.metadata.record_sets:
    rs_ids.append(rs['@id'])
    print(f"Found record set: {rs['@id']}")

dataframes = {}
for record_set_id in rs_ids:
    print(f"Loading records from {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# If only one main record set (as common for data tables), use the first
if rs_ids:
    print(f"Available columns in main DataFrame ({rs_ids[0]}):")
    print(dataframes[rs_ids[0]].columns.tolist())
    display(dataframes[rs_ids[0]].head())
else:
    print("No record sets found in metadata.")

## 4. Exploratory Data Analysis (EDA)

Apply filtering and transformations using field and column `@id`s. For example, perform analyses on Protein Molecular Weight (`MW`) or number of unique peptides, as relevant, using the appropriate Croissant field `@id` as shown in code below. You can inspect dataframes and columns above to pick valid field names for filtering and grouping.

In [ ]:
# --- Set your record set and field @id here (as found above) ---
record_set_id = rs_ids[0] if rs_ids else None
# Choose a numeric field, e.g., 'cr:mw' (molecular weight), adjust according to your exploration
numeric_field_id = None

# Try to auto-detect some numeric fields from the schema
schema_fields = {field['@id']: field for rs in dataset.metadata.record_sets for field in rs.get('field', [])}
possible_numeric_fields = [fid for fid, f in schema_fields.items() if f.get('dataType') in ['schema:Float', 'schema:Integer', 'schema:Number']]
print(f"Numeric field candidates: {possible_numeric_fields}")

# Default for demo: try commonly named IDs first
for candidate in ['cr:mw', 'cr:num_unique_peptides', 'cr:num_total_peptides', 'cr:coverage_percentage']:
    if candidate in dataframes[record_set_id].columns:
        numeric_field_id = candidate
        break
# If not found, use the first available numeric field
if not numeric_field_id and possible_numeric_fields:
    for candidate in possible_numeric_fields:
        if candidate in dataframes[record_set_id].columns:
            numeric_field_id = candidate
            break
if not numeric_field_id:
    raise RuntimeError('Could not detect a numeric field for field analysis.')

# Filter for values above a threshold (e.g., MW > 50,000 Da)
threshold = 50000
filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a key field, e.g., 'cr:description' or 'cr:accession' if available
group_field_id = None
for candidate in ['cr:description', 'cr:accession', 'cr:gene', 'cr:sample']:
    if candidate in filtered_df.columns:
        group_field_id = candidate
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field (e.g., molecular weight) and explore its relationship with other features.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(dataframes[record_set_id][numeric_field_id].dropna(), bins=40, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id} in {record_set_id}")
plt.tight_layout()
plt.show()

# Optionally, visualize relation with a secondary field (e.g., number of peptides vs. MW)
secondary_field_id = None
for candidate in ['cr:num_total_peptides', 'cr:num_unique_peptides']:
    if candidate in dataframes[record_set_id].columns:
        secondary_field_id = candidate
        break
if secondary_field_id:
    plt.figure(figsize=(7,5))
    sns.scatterplot(data=dataframes[record_set_id], x=numeric_field_id, y=secondary_field_id)
    plt.xlabel(numeric_field_id)
    plt.ylabel(secondary_field_id)
    plt.title(f"{numeric_field_id} vs {secondary_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- The FAIR^2 dataset was loaded using the Croissant schema and `mlcroissant` for robust, schema-driven analysis.
- Record sets and fields were referenced via their precise Croissant `@id`, supporting FAIR and reproducible workflows.
- Preliminary EDA showed distributions and highlighted proteins with higher molecular weight.
- Further biological or workflow insights may require community or domain-specific mapping of field IDs to biological meaning.

> All steps in this notebook can be easily adjusted to use other datasets described via Croissant, ensuring transparency and code portability.